# GeoGuard: Satellite Landslide Detection
## Notebook 01: Dataset Exploration & Band Analysis

This notebook explores the **Landslide4Sense** benchmark dataset to inspect:
1. Data structure, array dimensions, and value ranges.
2. Channel definitions across all 14 multi-sensor bands.
3. Why optical **Sentinel-2 B2, B3, B4, and B8** (10m resolution) are selected for the CNN.
4. Why non-satellite ALOS PALSAR terrain bands (Slope and DEM) are omitted from the initial model.
5. Ground-truth landslide segmentation masks and how pixel thresholds affect patch-level binary class balance.

In [ ]:
import os, sys
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import LandslideDatasetLoader, ALL_BANDS, DEFAULT_SELECTED_BAND_INDICES
from src.preprocessing import SatellitePreprocessor

print(f"Project root: {PROJECT_ROOT}")
print(f"Number of total available bands: {len(ALL_BANDS)}")

### 2. Inspecting the 14 Bands in Landslide4Sense

In [ ]:
band_table = []
for idx, name in enumerate(ALL_BANDS):
    selected = "YES (CNN Input)" if idx in DEFAULT_SELECTED_BAND_INDICES else "No"
    band_table.append({"Index": idx, "Band Name": name, "Selected for CNN": selected})

df_bands = pd.DataFrame(band_table)
print(df_bands.to_string(index=False))

### 3. Visualizing Satellite Patches & Ground Truth Masks

In [ ]:
raw_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')
loader = LandslideDatasetLoader(data_dir=raw_dir, pixel_threshold=100)
samples = loader.get_sample_paths('train')
sample_id, img_p, mask_p = samples[0]
with h5py.File(img_p, 'r') as f_img, h5py.File(mask_p, 'r') as f_mask:
    img_all = f_img['img'][:]
    mask = f_mask['mask'][:]

patch_4ch = img_all[:, :, [1, 2, 3, 7]]
rgb = SatellitePreprocessor.to_rgb_display(patch_4ch)
cir = SatellitePreprocessor.to_false_color_display(patch_4ch)
slope = img_all[:, :, 12]
dem = img_all[:, :, 13]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(rgb); axes[0].set_title(f"True-Color RGB (#{sample_id})"); axes[0].axis('off')
axes[1].imshow(cir); axes[1].set_title("False-Color CIR (NIR-R-G)"); axes[1].axis('off')
pos_px = int(np.sum(mask == 1))
axes[2].imshow(mask, cmap='gray'); axes[2].set_title(f"Mask ({pos_px} px, {pos_px/mask.size*100:.1f}%)"); axes[2].axis('off')
axes[3].imshow(slope, cmap='magma'); axes[3].set_title("ALOS PALSAR Slope"); axes[3].axis('off')
axes[4].imshow(dem, cmap='terrain'); axes[4].set_title("ALOS PALSAR DEM"); axes[4].axis('off')
plt.tight_layout()
plt.show()

### 4. Patch-Level Binary Labelling & Threshold Sensitivity Curve

In [ ]:
masks = []
for s_id, i_p, m_p in samples:
    with h5py.File(m_p, 'r') as f_m:
        masks.append(f_m['mask'][:])

thresholds = [1, 16, 50, 100, 164, 328, 500, 1000]
df_thresh = LandslideDatasetLoader.evaluate_threshold_distribution(masks, thresholds)
print(df_thresh.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.plot(df_thresh['threshold_pixels'], df_thresh['positive_ratio'] * 100, marker='o', color='crimson', label='% Landslide Patches')
plt.plot(df_thresh['threshold_pixels'], df_thresh['negative_ratio'] * 100, marker='s', color='navy', label='% Stable Patches')
plt.axvline(x=100, color='gray', linestyle='--', label='Selected Default Threshold (100 px = ~0.6%)')
plt.xlabel('Landslide Pixel Threshold (pixels per 128x128 patch)')
plt.ylabel('Percentage of Patches (%)')
plt.title('Threshold Sensitivity Curve: Effect on Class Distribution')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()